# 1.线性回归的简洁实现
上一节的 数据迭代器、损失函数、优化器和神经网络层等，在现代深度学习库都为我们实现了这些组件。

## 1.1 生成数据集


In [4]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

In [5]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2

features, labels = d2l.synthetic_data(true_w, true_b, 1000)

## 1.2 读取数据集
我们将features和labels作为API的参数传递，并通过数据迭代器指定batch_size。is_train表示是否希望数据迭代器对象在每轮内打乱数据。

In [6]:
def load_array(data_arrays, batch_size, is_train = True):
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays) # 将元组中的多个张量作为位置参数传入
    return data.DataLoader(dataset, batch_size, shuffle = is_train)
batch_size = 10
data_iter = load_array((features, labels), batch_size)
next(iter(data_iter))

[tensor([[-0.6888,  0.7465],
         [ 0.4919,  0.3402],
         [ 1.0053,  0.4391],
         [ 0.9053,  1.6671],
         [-0.2004, -0.3501],
         [ 0.6961, -0.6847],
         [ 0.5107, -0.3661],
         [-0.7402,  1.0424],
         [-0.3897,  2.2466],
         [ 0.5454,  0.3003]]),
 tensor([[ 0.2911],
         [ 4.0148],
         [ 4.7035],
         [ 0.3384],
         [ 4.9844],
         [ 7.9144],
         [ 6.4692],
         [-0.8086],
         [-4.2264],
         [ 4.2782]])]

## 1.3 定义模型
对于标准深度学习模型，我们可以使用框架的预定义好的层。这使得我们只需要关注使用哪些层来构造模型，而不必关注层的实现细节。我们先定义一个模型变量**net** ，它是一个**Sequential** 类的实例。Sequential类将多个层串联在一起。当给定输入数据时，**Sequential实例将数据传入第一层，然后将第一层的输出作为第二层的输入，依次类推。** </br>
在PyTorch中，全连接层在**Linear**类中定义。我们将两个参数传递到nn.Lineear中。第一个参数指定**输入形状**，即2.第二个参数指定**输出特征形状**，输出特征形状为单个标量，因此为1。

In [7]:
# nn是神经网络的缩写
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

## 1.4 初始化模型参数
在使用net之前，我们需要初始化模型参数，如在线性回归模型中的权重和偏置。在这里，我们指定每个权重参数应该从均值为0、标准差为0.01的正态分布中随机抽样，偏执参数将初始化为零。

我们在构造**nn.Linear**时指定输入和输出的尺寸一样，现在我们能直接访问参数以设定它们的初始值。我们通过net[0]选择网络中的第一层，然后使用**weight.data**和**bias.data**方法访问参数。我们还可以用使用替代方法normal_和fill_来重写参数值。

In [8]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

## 1.5 定义损失函数
计算均方误差损失使用的是**MSELoss**，其也称为**平方$L_2$范数** 。默认情况下，它返回所有样本损失的平均值。

In [9]:
loss = nn.MSELoss()

## 1.6 定义优化算法
小批量随机梯度下降算法是一种优化神经网络的标准工具，PyTorch在**optim**模块中实现了该算法的许多变体。当我们实例化一个SGD实例时，我们要**指定优化的参数（可通过net.parameters()从我们的模型中获得）** 以及优化算法所需的超参数字典。小批量随机梯度下降只需要设置lr的值，这里设置为0.03。

In [11]:
trainer = torch.optim.SGD(net.parameters(), lr = 0.03)

## 1.7 训练
通过深度学习框架的高级API来实现我们的模型只需要相对较少的代码。我们不必单独分配参数、不必定义我们的损失函数，也不必手动实现小批量随机梯度下降。当我们需要更复杂的模型时，高级API的优势将极大显现。当我们有了所有的基本组件，训练过程的代码与我们从零开始实现时的非常相似。

在每轮里，我们将完整遍历一次数据集(train_data)，不断地从中获取一个小批量的输入和相应的标签。对于每个小批量，我们会执行以下步骤。</br>
$\cdot$ 通过调用net(X)生成预测并计算损失l(前向传播)。</br>
$\cdot$ 通过反向传播来计算梯度。</br>
$\cdot$ 通过调用优化器来更新模型参数。</br>

In [15]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y) # net(X)就是向前传播
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss{l:f}')

epoch 1, loss0.000093
epoch 2, loss0.000093
epoch 3, loss0.000093


In [17]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b -b)

w的估计误差： tensor([ 1.7476e-04, -5.0068e-05])
b的估计误差： tensor([7.1049e-05])
